> ⚠️ **Sustituido por `analytics/sentiment/batch_inference.py`** (punto 4 de la revisión de código).
>
> Lo que hacía este notebook — reseñas de TripAdvisor + Booking desde 2022, modelo
> `nlptown/bert-base-multilingual-uncased-sentiment`, cruce H3 del establecimiento y volcado
> incremental por bloques a `gold.nlp_sentimiento_resenas` — está portado tal cual al script:
>
> ```bash
> python analytics/sentiment/batch_inference.py --source resenas
> ```
>
> El script es el que ejecuta Airflow (`sentiment_batch_inference_resenas` en
> `dag_social_refresh` y `dag_historical_full`). **No ejecutes este notebook a la vez que el
> script**: los dos escriben en la misma tabla y la tabla no tiene UNIQUE, así que duplicarías filas.
>
> Se conserva como documentación de la Tarea 2.1 y del análisis original.


# TFM Tenerife — Tarea 2.1: Sentimiento general por reseña (BERT Multilingüe)

Analiza el sentimiento de las reseñas de TripAdvisor y Booking (2022 en adelante) con `nlptown/bert-base-multilingual-uncased-sentiment`, las cruza espacialmente con la malla H3, y sube el resultado a `gold.nlp_sentimiento_resenas`.

**Sobre el modelo:** no hace falta descargarlo a mano — la celda del Paso 3 lo descarga solo la primera vez que se ejecuta, directamente desde Hugging Face.

**Diseño resistente a cortes:** el resultado se va guardando en `gold` por bloques (cada ~320 reseñas), no todo junto al final. Si el proceso se interrumpe (se va la luz, se cierra el portátil...), solo se pierde el bloque que estaba a medias — vuelve a ejecutar el notebook entero y, gracias al filtro incremental del Paso 4, continuará justo donde se quedó, sin repetir ni duplicar nada.

## Paso 1 — Instalar dependencias

In [1]:
!pip install -q transformers torch sentencepiece sqlalchemy psycopg2-binary python-dotenv pandas

## Paso 2 — Conectar

In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd

load_dotenv()
engine = create_engine(os.environ['AZURE_DB_URL'], pool_pre_ping=True, pool_recycle=280)

with engine.connect() as conn:
    version = conn.execute(text('SELECT postgis_version();')).scalar()
print('Conectado. PostGIS:', version)

Conectado. PostGIS: 3.6 USE_GEOS=1 USE_PROJ=1 USE_STATS=1


## Paso 3 — Cargar el modelo (se descarga solo, la primera vez)

In [3]:
import torch
from transformers import pipeline

dispositivo = 0 if torch.cuda.is_available() else -1  # 0 = GPU, -1 = CPU
print('Usando', 'GPU' if dispositivo == 0 else 'CPU (sera mas lento)')

sentiment_pipe = pipeline(
    'text-classification',
    model='nlptown/bert-base-multilingual-uncased-sentiment',
    device=dispositivo,
)
print('Modelo cargado.')

Usando CPU (sera mas lento)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo cargado.


## Paso 4 — Extraer reseñas NUEVAS de TripAdvisor + Booking (2022 en adelante)

Se unen las dos fuentes y se excluye lo que ya tenga resultado en `gold.nlp_sentimiento_resenas` -- así no hace falta esperar a que la recopilación de datos esté completa: cada vez que se relance, solo procesa lo que sea nuevo desde la última vez, sin duplicar ni repetir trabajo ya hecho.

In [4]:
with engine.begin() as conn:
    conn.execute(text('''
        CREATE TABLE IF NOT EXISTS gold.nlp_sentimiento_resenas (
            resena_id text,
            hotel_id text,
            score integer,
            h3_index text,
            fuente text
        )
    '''))
print('Tabla gold.nlp_sentimiento_resenas lista (creada si no existia).')

Tabla gold.nlp_sentimiento_resenas lista (creada si no existia).


In [5]:
consulta_resenas = '''
    SELECT 'tripadvisor' AS fuente, r.review_id::text AS resena_id, r.location_id::text AS hotel_id, r.texto AS review_text
    FROM silver.tripadvisor_resenas r
    WHERE r.texto IS NOT NULL AND EXTRACT(YEAR FROM r.fecha_publicacion) > 2021
      AND NOT EXISTS (
          SELECT 1 FROM gold.nlp_sentimiento_resenas g
          WHERE g.resena_id = r.review_id::text AND g.fuente = 'tripadvisor'
      )

    UNION ALL

    SELECT 'booking' AS fuente, b.review_id AS resena_id, b.establishment_id AS hotel_id, b.review_text
    FROM silver.silver_booking_reviews b
    WHERE b.review_text IS NOT NULL AND EXTRACT(YEAR FROM b.review_date) > 2021
      AND NOT EXISTS (
          SELECT 1 FROM gold.nlp_sentimiento_resenas g
          WHERE g.resena_id = b.review_id AND g.fuente = 'booking'
      );
'''

df_resenas = pd.read_sql(consulta_resenas, engine)
print('Reseñas NUEVAS a analizar esta vez:', len(df_resenas))
print(df_resenas['fuente'].value_counts())

Reseñas NUEVAS a analizar esta vez: 56594
fuente
booking        55787
tripadvisor      807
Name: count, dtype: int64


## Paso 5 — Cruce espacial con la malla H3 (una sola vez, antes de procesar)

TripAdvisor ya tiene geometría (`tripadvisor_ubicaciones.geometry`, EPSG:32628). Booking solo tiene `latitude`/`longitude`, así que se construye la geometría al vuelo y se reproyecta para que coincida con la malla.

In [6]:
with engine.connect() as conn:
    srid_h3 = conn.execute(text('SELECT ST_SRID(geometry) FROM silver.silver_h3_grid LIMIT 1;')).scalar()
print('SRID de la malla H3:', srid_h3)

consulta_cruce_h3 = f'''
    WITH establecimientos_geo AS (
        SELECT location_id::text AS hotel_id, ST_Transform(geometry, {srid_h3}) AS geometry
        FROM silver.tripadvisor_ubicaciones

        UNION ALL

        SELECT establishment_id AS hotel_id,
               ST_Transform(ST_SetSRID(ST_MakePoint(longitude, latitude), 4326), {srid_h3}) AS geometry
        FROM silver.silver_booking_establishments
        WHERE latitude IS NOT NULL AND longitude IS NOT NULL
    )
    SELECT e.hotel_id, h.h3_index
    FROM establecimientos_geo e
    JOIN silver.silver_h3_grid h ON ST_Contains(h.geometry, e.geometry);
'''

df_hotel_h3 = pd.read_sql(consulta_cruce_h3, engine)
print('Establecimientos con hexagono asignado:', len(df_hotel_h3))

SRID de la malla H3: 4326
Establecimientos con hexagono asignado: 3545


## Paso 6 — Procesar y guardar por bloques (resistente a cortes)

Cada bloque de ~320 reseñas (10 lotes de 32) se analiza, se cruza con su hexágono, y se sube a `gold` **inmediatamente** -- no se espera al final. Imprime el ritmo real y una estimación del tiempo que falta, para que puedas decidir sobre la marcha si esperar, dejarlo corriendo de noche, o buscar acceso a GPU.

In [7]:
import time

TAMANO_LOTE = 32
FILAS_POR_CHECKPOINT = 320  # se guarda en gold cada ~10 lotes

total = len(df_resenas)
procesadas_total = 0
inicio_general = time.time()

if total == 0:
    print('No hay resenas nuevas que procesar -- todo lo disponible ya esta en gold.')
else:
    for inicio_chunk in range(0, total, FILAS_POR_CHECKPOINT):
        chunk = df_resenas.iloc[inicio_chunk:inicio_chunk + FILAS_POR_CHECKPOINT].copy()
        lista_textos = chunk['review_text'].tolist()

        resultados_chunk = []
        for i in range(0, len(lista_textos), TAMANO_LOTE):
            lote = lista_textos[i:i + TAMANO_LOTE]
            resultados_lote = sentiment_pipe(lote, batch_size=TAMANO_LOTE, truncation=True, max_length=512)
            resultados_chunk.extend(resultados_lote)

        chunk['score'] = [int(r['label'].split()[0]) for r in resultados_chunk]
        chunk = chunk.merge(df_hotel_h3, on='hotel_id', how='left')

        resultado_chunk = chunk[['resena_id', 'hotel_id', 'score', 'h3_index', 'fuente']]
        resultado_chunk.to_sql('nlp_sentimiento_resenas', engine, schema='gold', if_exists='append', index=False)

        procesadas_total += len(chunk)
        transcurrido = time.time() - inicio_general
        velocidad = procesadas_total / transcurrido if transcurrido > 0 else 0
        restantes = total - procesadas_total
        eta_min = (restantes / velocidad / 60) if velocidad > 0 else 0

        print(f'Guardadas {procesadas_total}/{total} -- {velocidad:.1f} resenas/seg -- estimado restante: {eta_min:.0f} min')

    with engine.begin() as conn:
        conn.execute(text(
            'CREATE INDEX IF NOT EXISTS idx_nlp_sentimiento_hotel_id '
            'ON gold.nlp_sentimiento_resenas (hotel_id)'
        ))
        conn.execute(text(
            'CREATE INDEX IF NOT EXISTS idx_nlp_sentimiento_h3_index '
            'ON gold.nlp_sentimiento_resenas (h3_index)'
        ))

    print()
    print('Completado. Total procesado y guardado en esta ejecucion:', procesadas_total)

Guardadas 323/56594 -- 2.0 resenas/seg -- estimado restante: 478 min
Guardadas 646/56594 -- 1.9 resenas/seg -- estimado restante: 486 min
Guardadas 966/56594 -- 2.1 resenas/seg -- estimado restante: 434 min
Guardadas 1286/56594 -- 2.3 resenas/seg -- estimado restante: 400 min
Guardadas 1606/56594 -- 2.3 resenas/seg -- estimado restante: 391 min
Guardadas 1926/56594 -- 2.5 resenas/seg -- estimado restante: 368 min
Guardadas 2246/56594 -- 2.5 resenas/seg -- estimado restante: 360 min
Guardadas 2566/56594 -- 2.6 resenas/seg -- estimado restante: 351 min
Guardadas 2886/56594 -- 2.6 resenas/seg -- estimado restante: 349 min
Guardadas 3206/56594 -- 2.7 resenas/seg -- estimado restante: 334 min
Guardadas 3526/56594 -- 2.6 resenas/seg -- estimado restante: 336 min
Guardadas 3846/56594 -- 2.6 resenas/seg -- estimado restante: 338 min
Guardadas 4166/56594 -- 2.6 resenas/seg -- estimado restante: 334 min
Guardadas 4486/56594 -- 2.6 resenas/seg -- estimado restante: 332 min
Guardadas 4806/56594 --

## Paso 7 — Verificar

In [8]:
with engine.connect() as conn:
    resumen = pd.read_sql('''
        SELECT fuente, COUNT(*) AS total, AVG(score) AS score_medio,
               COUNT(h3_index) AS con_hexagono
        FROM gold.nlp_sentimiento_resenas
        GROUP BY fuente
    ''', conn)

display(resumen)

,fuente,total,score_medio,con_hexagono
0,booking,55787,4.456934,54990
1,tripadvisor,813,3.627306,813


## Notas

- El ticket original solo menciona `resena_id`, `hotel_id`, `score` como columnas de salida. He añadido `h3_index` (para que el cruce espacial sirva de algo en la propia tabla) y `fuente` (para poder distinguir TripAdvisor de Booking después). Si el equipo prefiere la tabla más ajustada al ticket exacto, es fácil quitar estas dos columnas antes de subir.
- `hotel_id` incluye tanto hoteles como restaurantes de TripAdvisor (mismo nombre de columna que usa el ticket, aunque el contenido no sea solo hoteles).
- **Diseño incremental y resistente a cortes**: el Paso 4 excluye lo que ya tenga resultado en `gold`, y el Paso 6 guarda cada bloque de ~320 reseñas según lo va terminando. Si el proceso se corta a mitad, solo se pierde el bloque en curso -- vuelve a ejecutar el notebook entero (Pasos 1-7) y continuará justo donde se quedó, sin duplicar nada.
- No he podido ejecutar esto contra los datos ni el modelo reales. Si algo falla (nombre de columna, memoria insuficiente, etc.), pégame el error exacto.
- Con 55.787 reseñas de Booking (2022 en adelante) más las de TripAdvisor, en CPU esto puede tardar varias horas. El Paso 6 imprime el ritmo real y una estimación de tiempo restante nada más arrancar -- si ves que va a tardar demasiado, puedes interrumpirlo con tranquilidad (Kernel → Interrupt) y retomarlo más tarde sin perder el progreso ya guardado.